# 02 — SQA Physics: Trotter Slices, Transverse Field, and Schedules

**What you'll learn:**
- The physics behind Simulated Quantum Annealing
- What Trotter slices are and how many to use
- Why the transverse field Γ decays geometrically
- How `worldline_sweeps` and `cluster_sweeps` improve mixing
- How to read energy traces and diagnose convergence

> **Prerequisite**: `01_quickstart.ipynb`

In [ ]:
import os, sys
_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
_PY   = os.path.join(_ROOT, 'python')
if _PY not in sys.path:
    sys.path.insert(0, _PY)

import numpy as np
import matplotlib.pyplot as plt
from qanneal import (
    DenseIsing, SparseIsing, SparseEdge,
    solve, auto_schedule_sqa_tuned, SQASchedule,
    SQAAnnealer, SQAMetricsObserver
)

---

## 1 — The Physics of SQA (No Equations Required)

### Classical SA: thermal fluctuations
Classical SA at temperature T randomly accepts uphill moves with probability $e^{-\Delta E / T}$.
At high T the system explores freely; as T → 0 it freezes.

**Problem**: thermal fluctuations are *isotropic* — they don't care about barrier shape.
A narrow-but-tall barrier (like a rare optimal solution surrounded by bad neighbours)
is very hard to cross thermally.

### SQA: quantum tunneling through barriers
Real quantum annealers apply a **transverse magnetic field** $\Gamma$ that induces quantum
tunneling. A spin can simultaneously be +1 and −1 (superposition).

**SQA simulates this** using the Suzuki–Trotter decomposition:
- Replicate the $n$-spin system $M$ times along an imaginary time axis.
- The $M$ copies are called **Trotter slices** and are coupled by $J_\perp = \frac{1}{2}\ln\frac{1}{\tanh(\beta\Gamma/M)}$.
- Flipping one spin through all slices (a **worldline**) corresponds to a quantum tunneling event.

As $\Gamma \to 0$, the Trotter slices align → classical frozen state.

```
Imaginary time ↑
slice M: s₀=+1  s₁=-1  s₂=+1  ...   ← classical copy M
  ...     ...    ...    ...
slice 1: s₀=+1  s₁=+1  s₂=-1  ...   ← classical copy 1
         ─────  ─────  ─────
spin:      0      1      2
```

---

## 2 — Test Problem: Frustrated 10-Spin Ring

A ring of spins with antiferromagnetic (AFM) couplings is **frustrated** — you cannot
satisfy all bonds simultaneously. This creates many near-degenerate ground states,
making it a good test case for tunneling.

In [ ]:
def make_afm_ring(n, J_afm=1.0, h_noise=0.1, seed=0):
    """Antiferromagnetic ring — frustrated for odd n."""
    rng = np.random.default_rng(seed)
    h = rng.uniform(-h_noise, h_noise, n)   # small random field breaks degeneracy
    J = np.zeros((n, n))
    for i in range(n):
        j = (i + 1) % n
        J[i, j] = J[j, i] = J_afm          # positive = antiferromagnetic
    return DenseIsing(h, J)

n = 15   # odd ring → strongly frustrated
ising = make_afm_ring(n, J_afm=1.0, h_noise=0.05, seed=7)

# Ground truth (brute force for n=15 is 2^15=32768 — fast enough)
best_E = float('inf')
for mask in range(1 << n):
    spins = np.array([1 if (mask >> i) & 1 else -1 for i in range(n)])
    E = ising.energy(spins.tolist())
    if E < best_E:
        best_E = E

print(f'AFM ring n={n}: ground state energy = {best_E:.4f}')

---

## 3 — Effect of `trotter_slices`

More slices → better quantum simulation (smaller Trotter error) but $\propto M$ more compute.
The Trotter coupling is: $J_\perp = \frac{1}{2}\ln\frac{1}{\tanh(\beta\Gamma/M)}$

At typical (β, Γ) values, $M=16$ is already accurate for most problems. $M=32$ is the default.

In [ ]:
schedule = auto_schedule_sqa_tuned(ising, mode='balanced')
READS = 50

slice_counts = [4, 8, 16, 32, 64]
results_by_slices = {}

for M in slice_counts:
    r = solve(
        ising, method='sqa',
        reads=READS, sweeps_per_beta=50, worldline_sweeps=4,
        trotter_slices=M, schedule=schedule, seed=0, progress=False,
    )
    results_by_slices[M] = r
    hit_rate = np.mean([abs(e - best_E) < 0.5 for e in r.energies])
    print(f'  M={M:3d}: best_E={r.best_energy:.4f}  hit_rate={hit_rate:.2f}  mean={np.mean(r.energies):.3f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Mean energy vs Trotter slices
ax = axes[0]
mean_Es  = [np.mean(results_by_slices[M].energies) for M in slice_counts]
best_Es  = [results_by_slices[M].best_energy        for M in slice_counts]
ax.plot(slice_counts, mean_Es, 'o-', label='mean energy', color='#4c8bf5')
ax.plot(slice_counts, best_Es, 's--', label='best energy', color='#c84b31')
ax.axhline(best_E, color='green', linestyle=':', label='ground state', lw=1.5)
ax.set_xlabel('trotter_slices (M)')
ax.set_ylabel('Energy')
ax.set_title('Effect of Trotter slices')
ax.legend()

# Hit rate vs Trotter slices
ax2 = axes[1]
hit_rates = [np.mean([abs(e - best_E) < 0.5 for e in results_by_slices[M].energies])
             for M in slice_counts]
ax2.bar([str(M) for M in slice_counts], hit_rates, color='#4c8bf5')
ax2.set_xlabel('trotter_slices (M)')
ax2.set_ylabel('Hit rate (|E - E_gs| < 0.5)')
ax2.set_title(f'Ground-state hit rate ({READS} reads)')
ax2.set_ylim(0, 1)

fig.tight_layout()
plt.show()

print('\n→ M=16–32 is usually enough; diminishing returns beyond that.')

---

## 4 — Effect of Γ Schedule: Geometric vs Linear Decay

The transverse field $\Gamma$ controls how much quantum tunneling happens.
- **Start high** (Γ_start ≫ 1): system explores freely via tunneling.
- **End near zero** (Γ_end ≈ 0.01): system freezes into a classical state.

**Linear** decay wastes steps at the ineffective mid-Γ region.
**Geometric** decay spends more steps where tunneling is active (high Γ).

In [ ]:
steps = 80
gamma_start, gamma_end = 5.0, 0.01
beta_start, beta_end   = 0.2, 6.0

# Linear gamma
sch_linear = SQASchedule.from_vectors(
    betas  = np.linspace(beta_start, beta_end, steps).tolist(),
    gammas = np.linspace(gamma_start, gamma_end, steps).tolist(),
)

# Geometric gamma (tuned default)
sch_geom = SQASchedule.from_vectors(
    betas  = np.linspace(beta_start, beta_end, steps).tolist(),
    gammas = np.geomspace(gamma_start, gamma_end, steps).tolist(),
)

# Adaptive (problem-tuned)
sch_tuned = auto_schedule_sqa_tuned(ising, mode='balanced', steps=steps)

# Visualise schedules
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

t = np.arange(steps)
for sch, label, color in [(sch_linear, 'Linear Γ',   '#4c8bf5'),
                           (sch_geom,   'Geometric Γ', '#c84b31'),
                           (sch_tuned,  'Tuned Γ',     '#2ca02c')]:
    axes[0].plot(sch.gammas, label=label, color=color)
    axes[1].semilogy(sch.gammas, label=label, color=color)

axes[0].set_xlabel('Schedule step'); axes[0].set_ylabel('Γ')
axes[0].set_title('Gamma schedule (linear scale)')
axes[0].legend()

axes[1].set_xlabel('Schedule step'); axes[1].set_ylabel('Γ (log scale)')
axes[1].set_title('Gamma schedule (log scale) — geometric is a straight line')
axes[1].legend()

fig.tight_layout()
plt.show()

In [ ]:
# Compare solution quality
READS = 50
results_sch = {}
for name, sch in [('Linear Γ', sch_linear), ('Geometric Γ', sch_geom), ('Tuned Γ', sch_tuned)]:
    r = solve(
        ising, method='sqa',
        reads=READS, sweeps_per_beta=50, worldline_sweeps=4,
        trotter_slices=24, schedule=sch, seed=0, progress=False,
    )
    results_sch[name] = r
    hit = np.mean([abs(e - best_E) < 0.5 for e in r.energies])
    print(f'{name:14s}: best={r.best_energy:.4f}  hit_rate={hit:.2f}')

---

## 5 — Effect of `worldline_sweeps` and `cluster_sweeps`

### Worldline sweeps
A worldline sweep flips **one spin across all Trotter slices simultaneously**.
This is the discrete-time analog of a worldline kink in CT-PIMC.
It allows the system to flip a spin without paying the time-direction coupling cost
(because all slices flip together).

**Effect**: helps escape configurations where a spin is stuck in one state across all slices.

### Cluster sweeps
Swendsen–Wang-style cluster updates along the imaginary time axis.
Bonds between adjacent time-slices (with matching spins) are included in clusters.

**Effect**: dramatically improves mixing in strongly-frustrated or dense problems.

In [ ]:
schedule_tuned = auto_schedule_sqa_tuned(ising, mode='balanced')
READS = 60

configs = [
    ('No extras',           dict(worldline_sweeps=0, cluster_sweeps=0)),
    ('worldline=2',         dict(worldline_sweeps=2, cluster_sweeps=0)),
    ('worldline=5',         dict(worldline_sweeps=5, cluster_sweeps=0)),
    ('worldline=5, cluster=1', dict(worldline_sweeps=5, cluster_sweeps=1)),
]

print('Configuration           best_E    hit_rate   mean_E')
print('-' * 60)
for label, kwargs in configs:
    r = solve(
        ising, method='sqa',
        reads=READS, sweeps_per_beta=40,
        trotter_slices=24, schedule=schedule_tuned,
        seed=0, progress=False, **kwargs,
    )
    hit = np.mean([abs(e - best_E) < 0.5 for e in r.energies])
    print(f'{label:26s} {r.best_energy:8.4f}  {hit:.2f}       {np.mean(r.energies):.3f}')

---

## 6 — Using SQAMetricsObserver for Diagnostics

The observer records per-step energy and magnetization traces without altering the algorithm.

In [ ]:
from qanneal import SQAAnnealer, SQAMetricsObserver

obs = SQAMetricsObserver()

annealer = SQAAnnealer(
    ising,
    schedule_tuned,
    trotter_slices=24,
    replicas=1,
)
annealer.set_seed(0)

res = annealer.run(
    sweeps_per_beta=50,
    worldline_sweeps=5,
    cluster_sweeps=1,
    observer=obs,
)

print(f'Best energy found: {res.best_energy:.4f}  (ground state: {best_E:.4f})')
print(f'Observer captured {len(obs.energy_trace)} steps')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

steps_arr = np.arange(len(obs.energy_trace))
gammas_arr = np.array(schedule_tuned.gammas)
betas_arr  = np.array(schedule_tuned.betas)

# Energy trace with schedule overlay
ax = axes[0]
ax.plot(steps_arr, obs.energy_trace, color='#4c8bf5', lw=1.5, label='Energy')
ax.axhline(best_E, color='green', linestyle='--', lw=1.5, label='Ground state')
ax.set_xlabel('Schedule step')
ax.set_ylabel('Energy', color='#4c8bf5')
ax.set_title('SQA energy trace')
ax2_r = ax.twinx()
ax2_r.plot(steps_arr, gammas_arr, color='#c84b31', lw=1, linestyle='--', alpha=0.7, label='Γ')
ax2_r.set_ylabel('Γ (transverse field)', color='#c84b31')
ax.legend(loc='upper right')

# Magnetization trace
ax3 = axes[1]
ax3.plot(steps_arr, obs.magnetization_trace, color='#c84b31', lw=1.5)
ax3.axhline(0, color='gray', linestyle=':', lw=1)
ax3.set_xlabel('Schedule step')
ax3.set_ylabel('Magnetization = <s> = Σsᵢ/n')
ax3.set_title('Magnetization trace (0 = balanced spins)')

fig.tight_layout()
plt.show()

---

## 7 — SQA on a Sparse Problem: Max-Cut

For **sparse** problems use `SparseIsing` — O(|E|) memory and O(degree) per spin flip.

In [ ]:
# Max-Cut on a random 20-node graph with degree ≈ 4
# Ising encoding: J_ij = -1 for all edges (minimise = maximise cut)
rng = np.random.default_rng(42)
n_nodes = 20
degree  = 4

edge_set = {}
for i in range(n_nodes):
    for _ in range(degree):
        j = int(rng.integers(0, n_nodes - 1))
        if j >= i:
            j += 1
        key = (min(i, j), max(i, j))
        edge_set[key] = -1.0   # ferromagnetic coupling → favours cut

h = np.zeros(n_nodes)
edges = [SparseEdge(i, j, v) for (i, j), v in edge_set.items()]
sparse_ising = SparseIsing(h, edges, n_nodes)

print(f'MaxCut graph: n={n_nodes} nodes, {len(edges)} edges, avg_degree={2*len(edges)/n_nodes:.1f}')

# Solve with SQA using SparseIsing
sch = auto_schedule_sqa_tuned(sparse_ising, mode='balanced')

r_sparse = solve(
    sparse_ising, method='sqa',
    reads=40, sweeps_per_beta=60, worldline_sweeps=5,
    trotter_slices=24, schedule=sch, seed=0, progress=False,
)

# Count cut edges
spins = r_sparse.best_sample
cut_size = sum(1 for (i, j) in edge_set if spins[i] != spins[j])
print(f'Max-Cut found: {cut_size} / {len(edges)} edges cut  (best energy: {r_sparse.best_energy:.3f})')

---

## 8 — Parameter Tuning Cheatsheet

| Parameter | Start here | If not converging |
|-----------|-----------|------------------|
| `trotter_slices` | 16–32 | Try 48–64 |
| `sweeps_per_beta` | 40–60 | Double it |
| `worldline_sweeps` | 3–5 | Try 8–10 |
| `cluster_sweeps` | 0 | Enable (=1) for dense/frustrated problems |
| `reads` | 20–50 | More reads always helps; use multiprocessing |
| `mode` | `balanced` | Try `accurate` |

**Signs of good convergence:**
- Energy trace drops smoothly and plateaus before the schedule ends.
- Energy distribution across reads is tight (low std).
- Magnetization stabilises near a fixed value.

**Next**: `03_sqapt_and_ctpimc.ipynb` — replica exchange and continuous-time PIMC.